# Python Multithreading vs Multiprocessing vs AsyncIO

Python offers many different mechanisms to write concurrent code, but which is best?

To compare, we take a toy web crawler, make it concurrent by applying multithreading, multi-processing and asyncio coroutines, and compare performance.

Our baseline is the following non-concurrent, recursive implementation:


```python
def crawl(root: str) -> None:
    seen: set[str] = set()
    with requests.Session() as session:
        def crawl(url: str) -> None:
            print(f'processing {url}')
            html = session.get(url).content
            soup = BeautifulSoup(html)
            paths = {urljoin(url, a.get('href')) for a in soup.find_all('a')}
            for u in filter(lambda p: p.startswith(root) and p.endswith('/'), paths - seen):
                seen.add(u)
                crawl(u)

        crawl(root)
```

In [ ]:
import pandas

pandas.set_option('display.max_rows', 500)

df = pandas.read_csv('results.csv')
df = df.sort_values(by=['prog'])
df['speed'] = df['urls'] / df['time']
df = df.sort_values(by=['prog', 'concurrency'])

# Pick the highest speed for each prog and concurrency combination:
df = df.groupby(['concurrency', 'prog'])[['speed']].max()
df = df.reset_index()
max_x = int(df['concurrency'].max() * 1.1)
max_y = int(df['speed'].max() * 1.2)
df = df.set_index('concurrency')
df = df.pivot_table('speed', ['concurrency'], 'prog')
df = df.interpolate()

ax = df.plot(grid=True, xlim=(0, max_x), ylim=(0, max_y), title='Python web crawler concurrency benchmark', ylabel='pages/sec')
# A dotted line at 16, the number of logical CPU cores during the benchmark:
ax.axvline(16, color="grey", linestyle="--")
ax

In [ ]:
display(df)